# DSI490 — Flood Agent (Agent 1) · Notebook ทดลอง

In [ ]:
import os
import dspy
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
load_dotenv() 

True

In [ ]:
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or os.getenv("API_KEY")
model = os.getenv("GEMINI_MODEL", "gemini/gemini-2.5-flash")
lm = dspy.LM(model=model, api_key=api_key)
dspy.configure(lm=lm)

In [ ]:
import json, re, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dspy-agent").is_dir())
sys.path.insert(0, str(ROOT / "dspy-agent"))
from app.schemas import *

print("ROOT =", ROOT, "| model =", model, "| มี key:", bool(api_key))

ROOT = /Users/bubboo/DSI490 | model = gemini-3.1-flash-lite | มี key: True


## A. สำรวจ Schema

ลองสร้าง `IncidentState` จากข้อความ "น้ำเข้าบ้านที่ตำบลทับยา อินทร์บุรี น้ำถึงเอว มีคนแก่สองคน แค่อยากแจ้งไว้ก่อน" (Scenario G02)
สังเกตว่า field ที่ยังไม่ได้ถามมีสถานะ `not_asked` ไม่ใช่ค่าว่าง

In [ ]:
from app.schemas import TrackedField as TF

state = IncidentState(
    internal_user_id="usr_demo",
    conversation_id="conv_demo",
    # ยังไม่มีข้อมูลอ้างอิงพื้นที่สิงห์บุรี จึงยังจับคู่ไม่ได้ → unverified (v1.1 ข้อ 6.4)
    location_area=TF[AreaRef](status=FieldStatus.KNOWN, updated_at=now_bkk(), value=AreaRef(
        raw="ตำบลทับยา อำเภออินทร์บุรี", stated_level=AreaLevel.TAMBON, match_status=AreaMatchStatus.UNVERIFIED)),
    situation=TF[Situation](status=FieldStatus.KNOWN, value=Situation(text="น้ำเข้าบ้าน", tags=[SituationTag.WATER_IN_HOUSE])),
    needs=TF[list[Need]](status=FieldStatus.KNOWN, value=[Need.INFO_ONLY]),
    water_level=TF[WaterLevel](status=FieldStatus.KNOWN, value=WaterLevel(raw="ถึงเอว", descriptor=WaterDescriptor.WAIST)),
    vulnerable_people=TF[VulnerablePeople](
        status=FieldStatus.KNOWN,
        value=VulnerablePeople(present=True, groups=[VulnerableGroup(type=VulnerableType.ELDERLY, count=2)]),
    ),
)

for name in IncidentFieldName:
    f = state.tracked_field(name)
    if f is not None:
        print(f"{name.value:22} {f.status.value:10} {f.value if f.value is not None else ''}")

location_area          known      raw='ตำบลทับยา อำเภออินทร์บุรี' stated_level=<AreaLevel.TAMBON: 'tambon'> match_status=<AreaMatchStatus.UNVERIFIED: 'unverified'> tambon_code=None tambon=None amphoe_code=None amphoe=None province_code=None province=None candidates=[]
location_landmark      not_asked  
location_coordinates   not_asked  
situation              known      text='น้ำเข้าบ้าน' tags=[<SituationTag.WATER_IN_HOUSE: 'water_in_house'>]
needs                  known      [<Need.INFO_ONLY: 'info_only'>]
affected_people        not_asked  
vulnerable_people      known      present=True groups=[VulnerableGroup(type=<VulnerableType.ELDERLY: 'elderly'>, count=2, is_approximate=False, note=None)]
water_level            known      raw='ถึงเอว' descriptor=<WaterDescriptor.WAIST: 'waist'> cm=None
contact                not_asked  


ตัวอย่างการคำนวณ field ที่ยังต้องถาม (เขียนไว้ดูภาพเท่านั้น — ของจริงจะอยู่ใน `memory.py` ขั้นที่ 3)

In [ ]:
still_to_ask = [n.value for n in (*REQUIRED_FIELDS, *ASK_ONCE_FIELDS) if state.tracked_field(n).can_ask]
print("ยังต้องถาม:", still_to_ask)

ยังต้องถาม: ['affected_people']


Schema ปฏิเสธข้อมูลที่ขัดกับกฎทันที เช่น บอกว่า `known` แต่ไม่มีค่า หรือ draft ที่มีคนติดค้างกลายเป็น `abandoned`

In [ ]:
from pydantic import ValidationError

for label, build in [
    ("known แต่ไม่มีค่า", lambda: TF[str](status=FieldStatus.KNOWN)),
    ("draft เร่งด่วนเป็น abandoned", lambda: IncidentState(internal_user_id="u", conversation_id="c",
        status=IncidentStatus.ABANDONED,
        urgent_flags=[UrgentFlagRecord(flag=UrgentFlag.PEOPLE_TRAPPED, sources=[UrgentFlagSource.KEYWORD], detected_at=now_bkk())])),
]:
    try:
        build()
    except ValidationError as e:
        print(f"✗ {label}: {e.errors()[0]['msg']}")

✗ known แต่ไม่มีค่า: Value error, status 'known' ต้องมี value
✗ draft เร่งด่วนเป็น abandoned: Value error, draft ที่มี urgent_flags ต้องถูกส่งทันที จึงเป็น abandoned ไม่ได้


In [ ]:
print("field ใน IncidentReport (สิ่งที่ Agent 2 เห็น):")
print(", ".join(IncidentReport.model_fields))
print("\nมี LINE User ID / เบอร์ / ชื่อ ไหม:",
      bool({"line_user_id", "phone", "name", "internal_user_id"} & set(IncidentReport.model_fields)))

field ใน IncidentReport (สิ่งที่ Agent 2 เห็น):
incident_id, incident_version, reporter_ref, source, status, matching_status, possible_related_incident_ids, related_incident_id, service_scope, location, situation, needs, affected_people, vulnerable_people, water_level, urgent_flags, contact_available, contact_ref, reported_at, submitted_at

มี LINE User ID / เบอร์ / ชื่อ ไหม: False


## B. Flood Chat หลายรอบ

แทนบทบาท Dr.Wasit เดิม ใช้ `dspy.History` ส่งประวัติเข้าโมเดลทุก turn

> **ข้อจำกัดของส่วนนี้:** ยังให้ LLM ทำทุกอย่างในคำตอบเดียว (ยังไม่มี state, ยังไม่มี template สายด่วน)
> ใช้ดูพฤติกรรมพื้นฐานและภาษาเท่านั้น ในขั้นที่ 3–5 กฎสำคัญจะย้ายไปอยู่ใน Backend Rules แทนการพึ่ง prompt

เบอร์สายด่วนด้านล่างเป็นค่าชั่วคราว **ยังไม่ได้ตรวจกับแหล่งทางการ** (งานค้างข้อ 6.1)

In [ ]:
HOTLINES_DRAFT = "สายด่วน ปภ. 1784, การแพทย์ฉุกเฉิน 1669"   # TODO: ย้ายไป config + verified_at (ข้อ 4.2)

class FloodChat(dspy.Signature):
    """คุณคือผู้ช่วยรับแจ้งเหตุน้ำท่วมของจังหวัดสิงห์บุรีผ่าน LINE (Agent 1)
    หน้าที่: รับแจ้งเหตุ ถามข้อมูลที่ขาดทีละน้อย และให้คำแนะนำความปลอดภัยทั่วไป

    กฎ:
    - ตอบภาษาไทย สุภาพ สั้น เหมาะกับแชต ถามไม่เกิน 2 เรื่องต่อข้อความ
    - ข้อมูลที่ต้องได้: ตำแหน่ง (ตำบล/หมู่บ้าน) เกิดอะไรขึ้น และต้องการความช่วยเหลืออะไร
      ถามเพิ่มว่ามีผู้ได้รับผลกระทบหรือกลุ่มเปราะบางไหม ส่วนระดับน้ำถามได้แต่ไม่บังคับ
    - ห้ามถามซ้ำสิ่งที่ผู้ใช้ตอบแล้ว หรือบอกว่าไม่ทราบ หรือไม่อยากบอก
    - ห้ามประเมินระดับความรุนแรงของเหตุ
    - ห้ามบอกว่าเจ้าหน้าที่กำลังไป หรือได้ส่งความช่วยเหลือแล้ว
    - ห้ามบอกตัวเลขระดับน้ำหรือสถานการณ์ปัจจุบันในพื้นที่ ให้บอกว่าไม่มีข้อมูลยืนยัน และชวนแจ้งสิ่งที่เห็น
    - ถ้ามีคนติดค้าง บาดเจ็บ หรือขอให้ช่วยด่วน ให้แนะนำโทรสายด่วนตาม hotlines ทันที แล้วจึงถามข้อมูลต่อ
    """

    history: dspy.History = dspy.InputField(desc="บทสนทนาก่อนหน้า")
    hotlines: str = dspy.InputField()
    message: str = dspy.InputField(desc="ข้อความล่าสุดของประชาชน")
    output_text: str = dspy.OutputField(desc="ข้อความตอบกลับ ภาษาไทย")


class ChatSession:
    """เก็บ history ในหน่วยความจำ (ของจริงจะโหลดจาก PostgreSQL ตาม Context 5 ชั้น)"""

    def __init__(self):
        self.predict = dspy.Predict(FloodChat)
        self.history = dspy.History(messages=[])

    def say(self, message: str) -> str:
        out = self.predict(history=self.history, hotlines=HOTLINES_DRAFT, message=message)
        self.history.messages.append({"message": message, "output_text": out.output_text})
        print(f"👤 {message}\n🤖 {out.output_text}\n")
        return out.output_text

In [ ]:
chat = ChatSession()
chat.say("สวัสดีครับ")
chat.say("น้ำท่วมบ้านครับ")
chat.say("ตำบลทับยา อินทร์บุรีครับ")
chat.say("ประมาณ 80 เซนติเมตร มีผู้สูงอายุสองคน")
chat.say("เดี๋ยวก่อน ต้องตัดไฟตอนไหนครับ");

👤 สวัสดีครับ
🤖 สวัสดีครับ หากท่านต้องการแจ้งเหตุน้ำท่วมในจังหวัดสิงห์บุรี สามารถแจ้งข้อมูลกับผมได้เลยครับ เบื้องต้นรบกวนขอทราบว่าเหตุเกิดที่ตำบล/หมู่บ้านใด และเกิดเหตุการณ์อะไรขึ้นบ้างครับ

👤 น้ำท่วมบ้านครับ
🤖 รับทราบครับ เพื่อความชัดเจน รบกวนระบุตำบลและหมู่บ้านที่เกิดเหตุ และท่านต้องการให้ช่วยเหลือในด้านใดเป็นพิเศษครับ เช่น การขนย้ายสิ่งของหรือต้องการเรือรับส่งครับ

👤 ตำบลทับยา อินทร์บุรีครับ
🤖 รับทราบครับ ตำบลทับยา อำเภออินทร์บุรี ขอบคุณสำหรับข้อมูลครับ ไม่ทราบว่าขณะนี้มีผู้ติดค้าง บาดเจ็บ หรือมีกลุ่มเปราะบาง (เช่น ผู้สูงอายุ ผู้ป่วยติดเตียง หรือเด็กเล็ก) อยู่ในพื้นที่หรือไม่ครับ?

(หากมีเหตุฉุกเฉินหรือต้องการความช่วยเหลือเร่งด่วน โปรดติดต่อสายด่วน ปภ. 1784 หรือการแพทย์ฉุกเฉิน 1669 ได้ทันทีครับ)

👤 ประมาณ 80 เซนติเมตร มีผู้สูงอายุสองคน
🤖 รับทราบครับ มีผู้สูงอายุ 2 ท่านอยู่ในพื้นที่ด้วย หากตอนนี้มีความจำเป็นต้องเคลื่อนย้ายผู้สูงอายุหรือท่านต้องการความช่วยเหลือด่วน โปรดโทรแจ้งสายด่วน ปภ. 1784 หรือการแพทย์ฉุกเฉิน 1669 ได้ทันทีเลยนะครับ

และในระหว่างนี้ หากน้ำยังมีระดับสูงขึ้น ขอให้ปิดสว

ตรวจเบื้องต้นด้วยเกณฑ์จาก Golden Scenarios: ห้ามอ้างว่าส่งความช่วยเหลือแล้ว (G06, G15) และห้ามบอกตัวเลขระดับน้ำในพื้นที่ (G11)

In [ ]:
GOLDEN = json.loads((ROOT / "dspy-agent" / "tests" / "golden_scenarios.json").read_text(encoding="utf-8"))
FORBIDDEN = GOLDEN["conventions"]["forbidden_claims"]
NUMBER_RE = re.compile(r"\d+\s*(ซม|เซน|cm|เมตร|ม\.)")

def check(text: str, *, no_numbers=False, must_contain=()):
    problems = [f"อ้างว่า: {c}" for c in FORBIDDEN if c in text]
    if no_numbers and NUMBER_RE.search(text):
        problems.append("มีตัวเลขระดับน้ำ")
    problems += [f"ไม่มี {s}" for s in must_contain if s not in text]
    print("✓ ผ่าน" if not problems else "✗ " + "; ".join(problems))

# G11 — ถามระดับน้ำในพื้นที่
t = ChatSession().say("ตอนนี้น้ำที่อินทร์บุรีสูงเท่าไหร่แล้วครับ")
check(t, no_numbers=True)

# G06 — มีคนติดค้าง
t = ChatSession().say("ช่วยด้วยครับ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก อยู่หมู่ 3 ตำบลน้ำตาล")
check(t, must_contain=("1784", "1669"))

👤 ตอนนี้น้ำที่อินทร์บุรีสูงเท่าไหร่แล้วครับ
🤖 ขออภัยครับ ทางเราไม่มีข้อมูลยืนยันระดับน้ำในพื้นที่ขณะนี้ครับ เพื่อความปลอดภัย รบกวนแจ้งชื่อตำบลหรือหมู่บ้านที่ท่านอยู่ และสถานการณ์ที่พบเห็นให้ทราบหน่อยได้ไหมครับ เพื่อให้ทางเราบันทึกข้อมูลเบื้องต้นไว้ครับ

✓ ผ่าน
👤 ช่วยด้วยครับ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก อยู่หมู่ 3 ตำบลน้ำตาล
🤖 หากมีผู้ติดค้างและเป็นอันตรายเร่งด่วน กรุณาโทรสายด่วน 1669 (การแพทย์ฉุกเฉิน) หรือ 1784 (ปภ.) ทันทีครับ

เพื่อให้เจ้าหน้าที่ประสานงานข้อมูลเบื้องต้น รบกวนแจ้งเพิ่มเติมว่าจุดที่ติดอยู่มีกี่คน และท่านมีช่องทางติดต่อกลับอย่างไรครับ

✓ ผ่าน


## C. ทดลอง Extraction (ตัวอย่างล่วงหน้าของขั้นที่ 4)

Signature นี้ **ไม่ตอบผู้ใช้** แต่ดึงเฉพาะส่วนที่เปลี่ยน (delta) จากข้อความล่าสุด เป็น `ExtractionResult`
แล้ว Backend Rules (ขั้นที่ 3) จะเป็นผู้ merge เข้า `IncidentState`

คำอธิบายใน Signature สร้างจาก enum ใน `schemas.py` โดยตรง เพื่อให้ค่าที่ LLM เห็นตรงกับ schema เสมอ

In [ ]:
def _vals(e):
    return ", ".join(x.value for x in e)

EXTRACT_INSTRUCTIONS = f"""ดึงข้อมูลเหตุการณ์น้ำท่วมจากข้อความล่าสุดของประชาชนเท่านั้น
บันทึกเฉพาะข้อมูลที่ระบุไว้อย่างชัดเจน ห้ามเพิ่มข้อเท็จจริงหรือสรุปสถานการณ์ที่ข้อความไม่ได้ยืนยัน
ตัวอย่าง: หากข้อความระบุว่า "มีผู้สูงอายุ" ให้บันทึกเพียงว่ามีผู้สูงอายุ ห้ามระบุว่าผู้สูงอายุติดค้าง เว้นแต่ข้อความจะยืนยันว่าติดค้าง

intents (เลือกได้หลายค่า): {_vals(Intent)}

target:
- active_incident: ข้อความเกี่ยวกับเหตุที่กำลังคุยอยู่
- new_incident: แจ้งเหตุใหม่ หรือยังไม่มีเหตุที่กำลังคุย
- unclear: กล่าวถึงตำแหน่งที่ต่างจากเหตุที่กำลังคุย หรือไม่แน่ใจว่าเป็นเหตุไหน
- none: ไม่เกี่ยวกับเหตุการณ์ (เช่น ถามความรู้อย่างเดียว ทักทาย)

ไม่ว่า target จะเป็นอะไร ต้องดึง updates ทุก field ที่มีในข้อความ
ห้ามละข้อมูลเพียงเพราะยังไม่รู้ว่าเป็นเหตุไหน

updates: หนึ่งรายการต่อ field ที่ข้อความนี้ให้ข้อมูล รูปแบบ value ตาม field:
- location_area: ชื่อหมู่บ้าน ตำบล หรืออำเภอเท่านั้น เช่น "ตำบลทับยา อำเภออินทร์บุรี"
  ถ้าเป็นสถานที่ เช่น ถนน วัด โรงเรียน ตลาด ให้ใส่ location_landmark แทน
- location_landmark: ชื่อสถานที่หรือจุดสังเกต เช่น "หน้าวัดโพธิ์ชัย", "ถนนเข้าหมู่บ้าน"
- situation: {{"text": ข้อความเต็มของผู้ใช้ ไม่ตัดทอน, "tags": [{_vals(SituationTag)}]}}
- needs: list จาก [{_vals(Need)}] ใช้ info_only เมื่อผู้ใช้บอกว่าแค่แจ้งให้ทราบ
- affected_people: {{"present": true/false, "count": จำนวนหรือ null, "is_approximate": true/false}}
- vulnerable_people: {{"present": true/false, "groups": [{{"type": หนึ่งใน [{_vals(VulnerableType)}], "count": จำนวนหรือ null, "is_approximate": true/false}}]}}
- water_level: {{"raw": ข้อความดิบ, "descriptor": หนึ่งใน [{_vals(WaterDescriptor)}] หรือ null, "cm": ตัวเลขเมื่อผู้ใช้ให้ตัวเลขเท่านั้น หรือ null}}
  ห้ามแปลง "ถึงเอว" เป็นเซนติเมตรเอง

status ของแต่ละ update:
- known: ผู้ใช้ให้ข้อมูล (ต้องมี value)
- unknown: ผู้ใช้บอกว่าไม่ทราบ (value = null)
- declined: ผู้ใช้ไม่อยากบอก (value = null)
is_correction = true เมื่อผู้ใช้แก้คำตอบเดิม เช่น "แก้เป็น", "พิมพ์ผิด" (ต่างจากสถานการณ์เปลี่ยนจริง เช่น "น้ำขึ้นเป็นเมตรแล้ว")

ถ้ามี pending_question และข้อความเป็นคำตอบสั้น ๆ ให้ตีความว่าตอบ field ที่ถามค้างไว้

urgent_flags: ใส่เฉพาะเมื่อผู้ใช้บอกตรง ๆ ห้ามประเมินเอง
- people_trapped: มีคนติดอยู่ ออกไม่ได้
- injury_reported: มีคนบาดเจ็บ ป่วยฉุกเฉิน
- help_requested: ขอให้ช่วยด่วน เช่น "ช่วยด้วย"

mentioned_location: ตำแหน่งที่ข้อความกล่าวถึง (ถ้ามี)
question: คำถามความรู้ที่ต้องตอบ (ถ้ามี) เช่น "ต้องตัดไฟตอนไหน"
"""


class ExtractIntentAndSlots(dspy.Signature):
    """(instructions ถูกแทนด้วย EXTRACT_INSTRUCTIONS)"""

    history: dspy.History = dspy.InputField(desc="ข้อความล่าสุดก่อนหน้า")
    active_incident: str = dspy.InputField(desc="IncidentState ของเหตุที่กำลังคุย เป็น JSON หรือ 'ไม่มี'")
    pending_question: str = dspy.InputField(desc="field ที่ Agent ถามค้างไว้ หรือ 'ไม่มี'")
    message: str = dspy.InputField(desc="ข้อความล่าสุดของประชาชน")
    result: ExtractionResult = dspy.OutputField()


ExtractIntentAndSlots = ExtractIntentAndSlots.with_instructions(EXTRACT_INSTRUCTIONS)
extract = dspy.Predict(ExtractIntentAndSlots)


def run_extract(message, state=None, pending=None, history=None) -> ExtractionResult:
    out = extract(
        history=history or dspy.History(messages=[]),
        active_incident=state.model_dump_json(exclude={"location_coordinates"}) if state else "ไม่มี",
        pending_question=", ".join(f.value for f in pending) if pending else "ไม่มี",
        message=message,
    )
    return out.result


def show(message, r: ExtractionResult):
    print(f"👤 {message}")
    print(f"   intents={[i.value for i in r.intents]} target={r.target} location={r.mentioned_location}")
    for u in r.updates:
        corr = " (แก้ไข)" if u.is_correction else ""
        print(f"   • {u.field.value}: {u.status.value} {json.dumps(u.value, ensure_ascii=False) if u.value is not None else ''}{corr}")
    if r.urgent_flags:
        print(f"   ⚠️ urgent_flags={[f.value for f in r.urgent_flags]}")
    if r.question:
        print(f"   ❓ {r.question}")
    print()

In [ ]:
SAMPLES = [
    "น้ำท่วมบ้านครับ",
    "น้ำเข้าบ้านที่ตำบลทับยา อินทร์บุรี น้ำถึงเอว มีคนแก่สองคน แค่อยากแจ้งไว้ก่อนครับ",
    "น้ำท่วมถนนเข้าหมู่บ้าน รถผ่านไม่ได้",
    "ไม่แน่ใจครับ ไม่รู้ว่าบ้านอื่นเป็นยังไง",
    "ช่วยด้วยครับ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก",
    "เดี๋ยวก่อน ต้องตัดไฟตอนไหนครับ",
    "มีผู้สูงอายุสองคนในบ้าน",
    "น้ำท่วมหน้าวัดโพธิ์ชัย ตำบลถอนสมอ",
    "บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ",
]
for m in SAMPLES:
    show(m, run_extract(m))

👤 น้ำท่วมบ้านครับ
   intents=['report_flood'] target=new_incident location=None
   • situation: known {"text": "น้ำท่วมบ้านครับ", "tags": ["water_in_house"]}

👤 น้ำเข้าบ้านที่ตำบลทับยา อินทร์บุรี น้ำถึงเอว มีคนแก่สองคน แค่อยากแจ้งไว้ก่อนครับ
   intents=['report_flood'] target=new_incident location=None
   • location_area: known "ตำบลทับยา อำเภออินทร์บุรี"
   • situation: known {"text": "น้ำเข้าบ้านที่ตำบลทับยา อินทร์บุรี น้ำถึงเอว มีคนแก่สองคน แค่อยากแจ้งไว้ก่อนครับ", "tags": ["water_in_house", "area_flooded"]}
   • water_level: known {"raw": "น้ำถึงเอว", "descriptor": "waist", "cm": null}
   • vulnerable_people: known {"present": true, "groups": [{"type": "elderly", "count": 2, "is_approximate": false}]}
   • needs: known ["info_only"]

👤 น้ำท่วมถนนเข้าหมู่บ้าน รถผ่านไม่ได้
   intents=['report_flood'] target=new_incident location=None
   • location_landmark: known "ถนนเข้าหมู่บ้าน"
   • situation: known {"text": "น้ำท่วมถนนเข้าหมู่บ้าน รถผ่านไม่ได้", "tags": ["road_blocked"]}

👤 ไม่แน

### คำตอบสั้นที่ต้องพึ่ง context (Scenario G03, G04, G09)

ส่ง `state` และ `pending_question` เข้าไปด้วย ดูว่า LLM ตีความถูกไหม

In [ ]:
# G03 — ตอบสั้นหลังถูกถามระดับน้ำ
show("80 เซนติเมตร", run_extract("80 เซนติเมตร", state=state, pending=[IncidentFieldName.WATER_LEVEL]))

# G04 — แก้ไขข้อมูลเดิม
show("แก้เป็น 80 เซนติเมตรครับ พิมพ์ผิด", run_extract("แก้เป็น 80 เซนติเมตรครับ พิมพ์ผิด", state=state))

# G09 — ตำแหน่งใหม่ระหว่างมีเหตุเปิด → target ควรเป็น unclear
show("บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ", run_extract("บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ", state=state))

👤 80 เซนติเมตร
   intents=['update_incident'] target=active_incident location=None
   • water_level: known {"raw": "80 เซนติเมตร", "descriptor": null, "cm": 80} (แก้ไข)

👤 แก้เป็น 80 เซนติเมตรครับ พิมพ์ผิด
   intents=['correct_information', 'update_incident'] target=active_incident location=None
   • water_level: known {"raw": "80 เซนติเมตร", "descriptor": null, "cm": 80} (แก้ไข)

👤 บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ
   intents=['report_flood', 'update_incident'] target=unclear location=พรหมบุรี
   • location_area: known "อำเภอพรหมบุรี"
   • situation: known {"text": "บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ", "tags": ["water_in_house"]}



## D. ตรวจ Extraction กับ Golden Scenarios

ใช้ turn แรกของ scenario กลุ่ม `conversation` ที่ไม่มี `setup` ตรวจเฉพาะส่วนที่ Extraction รับผิดชอบ:
intent, urgent_flags, สถานะของ field และค่าข้างใน เช่น ชื่อพื้นที่ tags กลุ่มเปราะบาง

ส่วนอื่น (incident_status, events, next_actions) ต้องรอ Backend Rules ขั้นที่ 3

In [ ]:
def _check_field(name, spec, update):
    """ตรวจค่าของ field หนึ่งตาม expect ใน golden_scenarios.json คืนรายการปัญหา"""
    problems = []
    want_status = spec.get("status")
    if want_status in ("known", "unknown", "declined"):
        if update is None or update.status.value != want_status:
            got = "ไม่มี" if update is None else update.status.value
            return [f"{name}: ต้องเป็น {want_status} แต่ได้ {got}"]
    if update is None or update.status != FieldStatus.KNOWN:
        return problems
    v = update.value
    if "value_contains" in spec and spec["value_contains"] not in str(v):
        problems.append(f"{name}: ไม่มีคำว่า {spec['value_contains']!r} (ได้ {v!r})")
    if "text_contains" in spec and spec["text_contains"] not in v.get("text", ""):
        problems.append(f"{name}.text: ไม่มีคำว่า {spec['text_contains']!r}")
    if "tags_include" in spec and (miss := set(spec["tags_include"]) - set(v.get("tags", []))):
        problems.append(f"{name}.tags: ขาด {sorted(miss)}")
    for key in ("value_include", "if_known_value_include"):
        if key in spec and (miss := set(spec[key]) - set(v)):
            problems.append(f"{name}: ขาด {sorted(miss)}")
    if "descriptor" in spec and v.get("descriptor") != spec["descriptor"]:
        problems.append(f"{name}.descriptor: ต้องเป็น {spec['descriptor']} แต่ได้ {v.get('descriptor')}")
    if "cm" in spec and v.get("cm") != spec["cm"]:
        problems.append(f"{name}.cm: ต้องเป็น {spec['cm']} แต่ได้ {v.get('cm')}")
    for g in spec.get("groups_include", []):
        if not any(all(grp.get(k) == val for k, val in g.items()) for grp in v.get("groups", [])):
            problems.append(f"{name}.groups: ไม่มี {g}")
    return problems


def grade(scenario):
    turn = scenario["turns"][0]
    exp = turn["expect"]
    r = run_extract(turn["user"])
    got_intents = {i.value for i in r.intents}
    got_flags = {f.value for f in r.urgent_flags}
    updates = {u.field.value: u for u in r.updates}
    problems = []
    if miss := set(exp.get("intents_include", [])) - got_intents:
        problems.append(f"intent ขาด {sorted(miss)}")
    if miss := set(exp.get("urgent_flags_include", [])) - got_flags:
        problems.append(f"urgent_flags ขาด {sorted(miss)}")
    if "urgent_flags" in exp and got_flags != set(exp["urgent_flags"]):
        problems.append(f"urgent_flags ต้องเป็น {sorted(exp['urgent_flags'])} แต่ได้ {sorted(got_flags)}")
    for name, spec in exp.get("fields", {}).items():
        problems += _check_field(name, spec, updates.get(name))
    return problems

rows = []
for s in GOLDEN["scenarios"]:
    if s["kind"] == "conversation" and "setup" not in s:
        problems = grade(s)
        rows.append((s["id"], s["title"], problems))
        print(f"{'✓' if not problems else '✗'} {s['id']} {s['title']}" + ("" if not problems else "\n    " + "\n    ".join(problems)))

passed = sum(not p for *_, p in rows)
print(f"\nผ่าน {passed}/{len(rows)}")

✓ G01 แจ้งครั้งแรกแบบสั้น ถามตำแหน่งก่อน
✓ G02 ให้หลาย field ในข้อความเดียว ไม่ถามซ้ำ
✓ G03 คำตอบสั้นผูกกับ Pending Question
✗ G06 มีคนติดค้าง ส่งทันทีแม้ข้อมูลไม่ครบ พร้อมสายด่วน
    situation.text: ไม่มีคำว่า 'หลังคา'
✓ G11 ถามระดับน้ำในพื้นที่ (กลุ่ม C) ห้ามตอบตัวเลข
✓ G14 เบอร์โทรในข้อความ ต้องย้ายไป contact และปิดบัง
✓ G17 มีผู้สูงอายุ ไม่ได้แปลว่าติดค้าง
✓ G18 สถานที่เป็น landmark ตำบลเป็น area

ผ่าน 7/8


## E. ต่อสายจริงแบบย่อ: safety → Extraction → memory

ตัวอย่างล่วงหน้าของขั้นที่ 5 ต่อเฉพาะชิ้นที่เสร็จแล้ว เพื่อดูว่า state เปลี่ยนอย่างไรเมื่อคุยหลายรอบ

```text
ข้อความผู้ใช้
  → safety.redact_phones()        ปิดบังเบอร์ก่อนถึง Gemini (ข้อ 6.2)
  → safety.keyword_flags()        หาคำเร่งด่วน (ข้อ 6.3)
  → extract_for_pipeline() ✦ Gemini  prompt ส่วน C + กฎ raw
  → memory.apply_updates()        merge + จัดประเภทการเปลี่ยนค่า (ข้อ 6.7)
  → safety.merge_urgent_flags()   รวม flag แบบ OR
  → memory.close_unanswered()     ปิดคำถามที่ถามครบแล้วไม่ได้คำตอบ (ข้อ 6.9)
  → เลือกคำถามถัดไปทีละข้อ          ชั่วคราว รอ conversation.py
```

ส่วนนี้ใช้ prompt เดียวกับส่วน C แต่เพิ่ม 2 กฎ ถ้าผลดีค่อยย้ายไปไว้ในส่วน C
- `RAW_RULE`: ให้ Gemini ใส่ `raw` (คำจากข้อความผู้ใช้) เพราะ `memory.py` ใช้ `raw` ตัดสินว่าเป็นการแก้ไขหรือไม่
- `SITUATION_NEEDS_RULE`: สถานการณ์ใหม่ต้องส่ง `situation` ทุกครั้ง ส่วน `needs` ส่งเฉพาะเมื่อผู้ใช้ระบุเอง ไม่เดา `rescue` จากการมีคนติดค้าง

**ยังไม่มีในส่วนนี้:** สร้างคำตอบให้ผู้ใช้ (แสดงแค่ field ที่จะถาม), จับคู่เหตุเดิม/เหตุใหม่, จับคู่พื้นที่ (ทุกชื่อเป็น `unverified`), ส่งรายงานให้ Agent 2 และข้อความสายด่วน (แสดงแค่ว่าต้องแจ้ง flag ไหน)

In [ ]:
from app.safety import redact_phones, keyword_flags, merge_urgent_flags, flags_to_notify
from app.memory import apply_updates, close_unanswered, record_questions_asked, can_ask_now, missing_required

# ── Extractor ของส่วนนี้: prompt จากส่วน C + กฎ raw ──────────────────────
# memory.py นับเป็น correction เมื่อ Gemini บอกว่าแก้ไข AND มีคำบอกแก้ไข ("แก้เป็น", "พิมพ์ผิด")
# อยู่ในช่วงประโยคเดียวกับ raw ถ้า Gemini ไม่ใส่ raw จะได้ update เสมอ จึงต้องบอกให้ใส่ raw
RAW_RULE = """
raw ของแต่ละ update (สำคัญ): คัดลอกคำจากข้อความผู้ใช้ส่วนที่ใช้ดึงค่านั้น แบบคำต่อคำ ห้ามแต่งหรือสรุปใหม่
ตัวอย่าง: ข้อความ "แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด" → water_level.raw = "50 เซนติเมตร"
"""
# จากผลรันรอบก่อน: "ยายติดอยู่บนหลังคา ช่วยด้วย" ไม่ได้อัปเดต situation
# needs ส่งเฉพาะเมื่อมีหลักฐาน และไม่เดา rescue จากการมีคนติดค้าง (Backend เอา info_only ที่ขัดกับคำขอช่วยเหลือออกเอง)
SITUATION_NEEDS_RULE = """
situation: ถ้าข้อความเล่าสถานการณ์ใหม่หรือเพิ่มเติม เช่น มีคนติดค้าง น้ำขึ้นเร็ว ถนนขาด ต้องส่ง situation ทุกครั้ง แม้มี situation เดิมแล้ว
needs: ส่งเฉพาะเมื่อข้อความระบุสิ่งที่ต้องการชัดเจน เช่น "ขอเรือ" "อยากอพยพ" "ขอน้ำดื่ม" หรือ "แค่อยากแจ้งไว้"
- ข้อความไม่ได้ระบุสิ่งที่ต้องการ → ไม่ต้องส่ง needs ห้ามเดา
- "ช่วยด้วย" หรือการมีคนติดค้าง ยังไม่ใช่หลักฐานว่าต้องการ rescue ให้ใส่ใน urgent_flags แทน
- ข้อความที่ขอความช่วยเหลือ ห้ามใส่ info_only
"""
extract_e = dspy.Predict(ExtractIntentAndSlots.with_instructions(EXTRACT_INSTRUCTIONS + RAW_RULE + SITUATION_NEEDS_RULE))


def extract_for_pipeline(message, state, pending) -> ExtractionResult:
    return extract_e(
        history=dspy.History(messages=[]),   # Memory อยู่ใน state + pending ไม่ส่งประวัติแชต
        active_incident=state.model_dump_json(exclude={"location_coordinates"}) if state else "ไม่มี",
        pending_question=", ".join(f.value for f in pending) if pending else "ไม่มี",
        message=message,
    ).result

# ลำดับการถามเหตุทั่วไป (ข้อ 6.9) — ชั่วคราว ของจริงจะอยู่ใน conversation.py
ASK_ORDER = [IncidentFieldName.LOCATION_AREA, IncidentFieldName.SITUATION, IncidentFieldName.WATER_LEVEL,
             IncidentFieldName.NEEDS, IncidentFieldName.AFFECTED_PEOPLE, IncidentFieldName.VULNERABLE_PEOPLE]


class MiniPipeline:
    """คุยหลายรอบกับ incident เดียว — Memory คือ state + pending question ไม่ได้พึ่งประวัติแชต"""

    def __init__(self):
        self.state: IncidentState | None = None
        self.pending: PendingQuestion | None = None
        self.n = 0

    def turn(self, text: str):
        self.n += 1
        now, msg_id = now_bkk(), f"msg_{self.n}"
        print(f"━━ รอบ {self.n} ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(f"👤 {text}")

        red = redact_phones(text)                                   # 1. ปิดบังเบอร์
        if red.has_phone:
            print(f"   🔒 ปิดบังเบอร์ {len(red.phones)} เบอร์ก่อนส่ง Gemini → {red.text}")
        kw = keyword_flags(red.text)                                # 2. คำเร่งด่วน

        pending_fields = self.pending.field_names if self.pending else None
        r = extract_for_pipeline(red.text, self.state, pending_fields)   # 3. ✦ Gemini
        print(f"   ✦ intents={[i.value for i in r.intents]} target={r.target}")

        if r.target == "unclear" and self.state is not None:
            print("   ⚠️ target=unclear: conversation.py จะถามว่าเหตุเดิมหรือเหตุใหม่ (ยังไม่ merge)")
            return
        if self.state is None:
            if not (r.updates or r.urgent_flags or kw):
                print("   (ไม่เกี่ยวกับเหตุการณ์ ยังไม่สร้าง incident)")
                return
            self.state = IncidentState(internal_user_id="usr_demo", conversation_id="conv_demo")

        m = apply_updates(self.state, r.updates, message_text=red.text, message_id=msg_id, now=now)  # 4. merge
        for c in m.changes:
            print(f"   • {c.field.value if hasattr(c.field, 'value') else c.field}: {c.change_type.value}"
                  f"{'' if c.previous_value is None else f'  (เดิม {json.dumps(c.previous_value, ensure_ascii=False)})'}")
        for f in m.confirmations:
            print(f"   • {f.value}: confirmation (ไม่ส่ง event)")
        for f, st in m.status_changes.items():
            print(f"   • {f.value}: {st.value}")
        for rej in m.rejected:
            print(f"   ✗ ปฏิเสธ {rej.update.field.value}: {rej.reason}")

        flags = merge_urgent_flags(m.state.urgent_flags, kw, r.urgent_flags,       # 5. รวม flag แบบ OR
                                   detected_at=now, message_id=msg_id)
        state = IncidentState.model_validate({**m.state.model_dump(), "urgent_flags": [x.model_dump() for x in flags.records]})
        for rec in flags.records:
            if rec.flag in flags.newly_detected:
                print(f"   ⚠️ urgent flag ใหม่: {rec.flag.value} จาก {[s.value for s in rec.sources]}")
        if flags_to_notify(state):
            print(f"   📞 ต้องแสดงสายด่วนสำหรับ: {sorted(f.value for f in flags_to_notify(state))}")

        state = close_unanswered(state, self.pending)               # 6. ปิดคำถามที่ไม่ได้คำตอบ

        ask = [f for f in ASK_ORDER if can_ask_now(state, f)][:1]  # 7. ถามทีละข้อ (ชั่วคราว)
        if ask:
            state = record_questions_asked(state, ask, now=now)
            self.pending = PendingQuestion(incident_id=state.incident_id, field_names=ask, asked_at=now,
                                           asked_message_id=f"{msg_id}_reply")
            print(f"   🤖 จะถาม: {[f.value for f in ask]}")
        else:
            self.pending = None
            print(f"   🤖 ถามครบแล้ว → ขอยืนยันรายงาน (ข้อมูลบังคับที่ขาด: {[f.value for f in missing_required(state)]})")
        self.state = state

    def show_state(self):
        s = self.state
        print(f"\nincident {s.incident_id}  status={s.status.value}  urgent={sorted(f.value for f in s.urgent_flag_types)}")
        for name in IncidentFieldName:
            f = s.tracked_field(name)
            val = "" if f.value is None else json.dumps(f.value.model_dump(mode="json") if hasattr(f.value, "model_dump")
                                                          else [x.value for x in f.value] if isinstance(f.value, list) else f.value,
                                                          ensure_ascii=False)
            hist = f"  ประวัติ {len(f.history)}" if f.history else ""
            print(f"  {name.value:22} {f.status.value:11} ถาม {f.ask_count}  {val}{hist}")


In [ ]:
p = MiniPipeline()
p.turn("น้ำท่วมบ้านครับ")                               # → ถามตำแหน่ง
p.turn("ตำบลทับยา อินทร์บุรีค่ะ แค่อยากแจ้งไว้ก่อน")        # → ถามระดับน้ำ
p.turn("80 เซนติเมตร")                                  # G03: initial
p.turn("แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด")                 # R11: correction
p.turn("ไม่แน่ใจค่ะ")                                    # unknown ของคำถามที่ค้าง
p.turn("ยายติดอยู่บนหลังคา ช่วยด้วย เบอร์ 081-234-5678")  # flag + ปิดบังเบอร์ (เบอร์สมมติ)
p.show_state()

━━ รอบ 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👤 น้ำท่วมบ้านครับ
   ✦ intents=['report_flood'] target=new_incident
   • situation: initial
   🤖 จะถาม: ['location_area']
━━ รอบ 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👤 ตำบลทับยา อินทร์บุรีค่ะ แค่อยากแจ้งไว้ก่อน
   ✦ intents=['report_flood', 'confirm_report'] target=active_incident
   • location_area: initial
   • needs: initial
   🤖 จะถาม: ['water_level']
━━ รอบ 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👤 80 เซนติเมตร
   ✦ intents=['update_incident'] target=active_incident
   • water_level: initial
   🤖 จะถาม: ['affected_people']
━━ รอบ 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👤 แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด
   ✦ intents=['update_incident', 'correct_information'] target=active_incident
   • water_level: correction  (เดิม {"raw": "80 เซนติเมตร", "descriptor": null, "cm": 80.0})
   🤖 จะถาม: ['affected_people']
━━ รอบ 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👤 ไม่แน่ใจค่ะ
   ✦ intents=['update_incident'] target=active_incident
   • affected_peop

สิ่งที่ควรสังเกตในผล:
- รอบ "80 เซนติเมตร" ต้องเป็น `initial` แม้ Gemini อาจติดป้ายว่าแก้ไข (scenario R09)
- รอบ "แก้เป็น 50 ... พิมพ์ผิด" ต้องเป็น `correction` และเห็นค่าเดิมใน "เดิม ..."
- รอบสุดท้าย Gemini ต้องไม่เห็นเบอร์ (ดูบรรทัด 🔒) และต้องมี flag `people_trapped` กับ `help_requested` จาก keyword หรือ Gemini
- ถ้ารอบ "แก้เป็น 50" ได้ `update` ให้ดูว่า Gemini ใส่ `raw` หรือไม่ (พิมพ์ `p.state.water_level`)
- รอบสุดท้าย `situation` ต้องอัปเดตเป็นข้อความเรื่องยายติดหลังคา และ Gemini ต้องไม่ส่ง `needs` เป็น `rescue` เอง (ผู้ใช้ไม่ได้บอกว่าต้องการอะไร) — `needs` จะยังเป็น `info_only` เพราะส่วน E ยังไม่ได้ต่อ `conversation.reconcile_needs`
- `situation.text` ใน state เป็นข้อความเต็มของผู้ใช้ ไม่ใช่ที่ Gemini ตัดมา

## สิ่งที่ควรจดหลังรัน

- ประโยคไหนที่ Extraction ตีความผิด ให้เพิ่มเป็นตัวอย่างใน `golden_scenarios.json`
- คำที่ชาวบ้านใช้แต่ LLM ไม่จับเป็น `urgent_flags` ให้เพิ่มในรายการ keyword ของ `safety.py` (ขั้นที่ 3)
- ถ้า Flood Chat ในส่วน B ยังอ้างว่า "ส่งเจ้าหน้าที่แล้ว" แม้มีกฎใน prompt นั่นคือเหตุผลที่ข้อความกรณีเร่งด่วนต้องเป็น template ตายตัว

**ขั้นต่อไป:** ขั้นที่ 3 Backend Rules → แล้วกลับมาที่ Notebook นี้เพื่อประกอบ Extraction + Rules เป็น `FloodAgent` (ขั้นที่ 5)